In [5]:
# ===========================
# Phase III modeling: Base vs Cluster, Regressor + Classifier
# ===========================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, average_precision_score, classification_report
from lightgbm import LGBMRegressor, LGBMClassifier

# ---------------------------
# 0. Load data (already engineered)
# ---------------------------
df = pd.read_csv("/Users/KrisLiu/Downloads/patient_features_with_encounter_cluster.csv")

# Ensure types
df['encounter_start_dt'] = pd.to_datetime(df['encounter_start_dt'], errors='coerce')
df['encounter_end_dt']   = pd.to_datetime(df['encounter_end_dt'],   errors='coerce')


In [6]:
# ---------------------------
# 1. Targets
# ---------------------------
# A) Duration regression target (unchanged)
target_days = 'encounter_duration_days'

# B) Count target (choose one)
#    Option 1 (your original): past_num_encounters (historical)
target_count = 'past_num_encounters'
#    Option 2 (optional): current encounter procedure count
# target_count = 'proc_count'  # uncomment to switch

# C) Short vs Long classifier target
#    Define long-stay threshold in days (e.g., >= 3 days is long)
long_stay_threshold = 3.0
df['is_long_stay'] = (df[target_days] >= long_stay_threshold).astype(int)


In [7]:
# ---------------------------
# 2. Feature sets
#    Remove encounter_setting; keep encounter_medical_service (categorical).
#    Cluster version adds 'encounter_cluster'.
# ---------------------------
base_features = [
    'age', 'PATIENT_SEX', 'PATIENT_RACE_ETHNICITY',
    'encounter_medical_service',
    'past_num_encounters', 'past_total_procedures',
    'past_total_icd_codes', 'past_total_encounter_days'
    # DO NOT include encounter_setting
]

cluster_features = base_features + ['encounter_cluster']

# Sanity: keep only columns that exist
base_features    = [c for c in base_features    if c in df.columns]
cluster_features = [c for c in cluster_features if c in df.columns]


In [8]:
# ---------------------------
# 3. Train/Test split with GroupKFold (by patient)
#    We first create a single holdout split to report metrics consistently.
# ---------------------------
# Create groups
groups = df['patient_id'].astype(str)

# Single split (80/20) preserving groups
# To use GroupKFold for a single split, we can manually take the first fold as test
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(df, groups=groups))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

def build_preproc(feature_list):
    # separate categorical and numeric
    cat_cols = [c for c in feature_list if df[c].dtype == 'object']
    num_cols = [c for c in feature_list if c not in cat_cols]

    preproc = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', 'passthrough', num_cols)
        ],
        remainder='drop'
    )
    return preproc, cat_cols, num_cols

def fit_and_eval_reg(feature_list, y_col, label):
    preproc, cat_cols, num_cols = build_preproc(feature_list)
    model = LGBMRegressor(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    pipe = Pipeline([
        ('prep', preproc),
        ('lgbm', model)
    ])
    X_tr, y_tr = df_train[feature_list], df_train[y_col]
    X_te, y_te = df_test[feature_list],  df_test[y_col]

    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)

    mae = mean_absolute_error(y_te, pred)
    r2  = r2_score(y_te, pred)
    print(f"[{label}] Regression {y_col}: MAE={mae:.3f}, R²={r2:.3f}")
    return pipe, pred, {'MAE': mae, 'R2': r2}

def fit_and_eval_cls(feature_list, y_col, label):
    preproc, cat_cols, num_cols = build_preproc(feature_list)
    clf = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )
    pipe = Pipeline([
        ('prep', preproc),
        ('lgbm', clf)
    ])
    X_tr, y_tr = df_train[feature_list], df_train[y_col]
    X_te, y_te = df_test[feature_list],  df_test[y_col]

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:,1]
    pred  = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_te, proba)
    ap  = average_precision_score(y_te, proba)
    print(f"[{label}] Classifier {y_col}: AUC={auc:.3f}, PR-AUC={ap:.3f}")
    print(classification_report(y_te, pred, digits=3))
    return pipe, proba, {'AUC': auc, 'PR_AUC': ap}
    

In [9]:
# ---------------------------
# 4. Train & evaluate (Base vs Cluster)
# ---------------------------
print("Data shape:", df.shape)

# Regressors
base_reg_count, base_pred_count, base_count_m = fit_and_eval_reg(base_features, target_count,  "Base")
base_reg_days,  base_pred_days,  base_days_m  = fit_and_eval_reg(base_features, target_days,   "Base")

cluster_reg_count, cluster_pred_count, cluster_count_m = fit_and_eval_reg(cluster_features, target_count, "Cluster")
cluster_reg_days,  cluster_pred_days,  cluster_days_m  = fit_and_eval_reg(cluster_features, target_days,  "Cluster")

# Classifier (short vs long)
base_cls,    base_prob_long,    base_cls_m    = fit_and_eval_cls(base_features,   'is_long_stay', "Base")
cluster_cls, cluster_prob_long, cluster_cls_m = fit_and_eval_cls(cluster_features,'is_long_stay', "Cluster")


Data shape: (15116, 27)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001165 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 315
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 30
[LightGBM] [Info] Start training from score 0.740655
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[Base] Regression past_num_encounters: MAE=0.207, R²=0.634
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000955 seconds.
You can set `force_row_wise=true`

In [15]:
# ---------------------------
# 5. Service-level workload comparison (using predictions)
#    For days: use predicted duration; for count: use predicted count.
# ---------------------------
def agg_workload(df_sub, pred_count, pred_days):
    tmp = df_sub[['encounter_medical_service']].copy()
    tmp['pred_count'] = pred_count
    tmp['pred_days']  = pred_days
    grp = tmp.groupby('encounter_medical_service').agg(
        Patients=('pred_days','size'),
        Avg_pred_count=('pred_count','mean'),
        Avg_pred_days =('pred_days','mean')
    )
    grp['Total_workload_days'] = grp['Avg_pred_days'] * grp['Patients']
    return grp

base_work = agg_workload(df_test, base_pred_count,    base_pred_days)
clst_work = agg_workload(df_test, cluster_pred_count, cluster_pred_days)

comp = (base_work
        .join(clst_work, lsuffix="_base", rsuffix="_cluster")
        .assign(
            Delta_avg_days = lambda d: d['Avg_pred_days_cluster'] - d['Avg_pred_days_base'],
            Delta_total_days = lambda d: d['Total_workload_days_cluster'] - d['Total_workload_days_base']
        )
       ).sort_values('Total_workload_days_base', ascending=False)

print("\n=== Global metrics (test set) ===")
print("  Model    MAE_count   R2_count   MAE_days   R2_days   AUC(long)  PR-AUC(long)")
print(f"  Base     {base_count_m['MAE']:.3f}       {base_count_m['R2']:.3f}     {base_days_m['MAE']:.3f}    {base_days_m['R2']:.3f}     {base_cls_m['AUC']:.3f}      {base_cls_m['PR_AUC']:.3f}")
print(f"  Cluster  {cluster_count_m['MAE']:.3f}       {cluster_count_m['R2']:.3f}     {cluster_days_m['MAE']:.3f}    {cluster_days_m['R2']:.3f}     {cluster_cls_m['AUC']:.3f}      {cluster_cls_m['PR_AUC']:.3f}")

print("\n=== Workload comparison by medical service (Cluster vs Base) ===")
print(comp.reset_index().to_string(index=False))




=== Global metrics (test set) ===
  Model    MAE_count   R2_count   MAE_days   R2_days   AUC(long)  PR-AUC(long)
  Base     0.207       0.634     9.195    0.085     0.810      0.769
  Cluster  0.212       0.623     9.208    0.088     0.808      0.764

=== Workload comparison by medical service (Cluster vs Base) ===
           encounter_medical_service  Patients_base  Avg_pred_count_base  Avg_pred_days_base  Total_workload_days_base  Patients_cluster  Avg_pred_count_cluster  Avg_pred_days_cluster  Total_workload_days_cluster  Delta_avg_days  Delta_total_days
        Nursing - medical / surgical           1574             0.522406           13.760912              21659.676089              1574                0.521603              13.798640                 21719.059286        0.037728         59.383197
                     General surgery            184             0.413340            8.624613               1586.928747               184                0.413744               8.619234     